In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv('telco_churn_cleaned.csv')
conn = sqlite3.connect('churn.db')
df.to_sql('churn', conn, if_exists='replace', index=False)

def run_query(query):
    return pd.read_sql_query(query, conn)

print("Ready!")

Ready!


#  Churn count by Contract Type

In [2]:
q1 = """
SELECT Contract,
COUNT(*) as Total_Customers,
SUM(Churn) as Churned,
ROUND(AVG(Churn)*100, 2) as Churn_Rate
FROM churn
GROUP BY Contract
ORDER BY Churn_Rate DESC
"""
print(run_query(q1))

         Contract  Total_Customers  Churned  Churn_Rate
0  Month-to-month             3875     1655       42.71
1        One year             1473      166       11.27
2        Two year             1695       48        2.83


#  Average Monthly Charges Churned vs Stayed

In [3]:
q2 = """
SELECT Churn_Label,
ROUND(AVG(MonthlyCharges), 2) as Avg_Monthly_Charge,
ROUND(AVG(tenure), 2) as Avg_Tenure
FROM churn
GROUP BY Churn_Label
"""
print(run_query(q2))

  Churn_Label  Avg_Monthly_Charge  Avg_Tenure
0     Churned               74.44       17.98
1      Stayed               61.27       37.57


# Label Customer Risk Level

In [4]:
q3 = """
SELECT customerID,
Contract,
tenure,
MonthlyCharges,
CASE
    WHEN Contract = 'Month-to-month' AND tenure < 12 THEN 'High Risk'
    WHEN Contract = 'Month-to-month' AND tenure < 24 THEN 'Medium Risk'
    WHEN Contract = 'One year' THEN 'Low Risk'
    ELSE 'Very Low Risk'
END as Risk_Level
FROM churn
ORDER BY Risk_Level
"""
print(run_query(q3))

      customerID        Contract  tenure  MonthlyCharges     Risk_Level
0     7590-VHVEG  Month-to-month       1           29.85      High Risk
1     3668-QPYBK  Month-to-month       2           53.85      High Risk
2     9237-HQITU  Month-to-month       2           70.70      High Risk
3     9305-CDSKC  Month-to-month       8           99.65      High Risk
4     6713-OKOMC  Month-to-month      10           29.75      High Risk
...          ...             ...     ...             ...            ...
7038  9281-CEDRU        Two year      68           64.10  Very Low Risk
7039  9767-FFLEM  Month-to-month      38           69.50  Very Low Risk
7040  0639-TSIQW  Month-to-month      67          102.95  Very Low Risk
7041  2569-WGERO        Two year      72           21.15  Very Low Risk
7042  3186-AJIEK        Two year      66          105.65  Very Low Risk

[7043 rows x 5 columns]


# High Value Churned Customers

In [9]:
q4 = """
WITH High_Value AS (
    SELECT customerID,
    MonthlyCharges,
    tenure,
    Churn_Label
    FROM churn
    WHERE MonthlyCharges > 60
)
SELECT Churn_Label,
COUNT(*) as Total,
ROUND(AVG(MonthlyCharges),2) as Avg_Charge
FROM High_Value
GROUP BY Churn_Label
"""
print(run_query(q4))

  Churn_Label  Total  Avg_Charge
0     Churned   1379       87.05
1      Stayed   2746       87.01


# Rank Customers by Monthly Charges

In [8]:
q5 = """
SELECT customerID,
Contract,
MonthlyCharges,
Churn_Label,
RANK() OVER (PARTITION BY Contract ORDER BY MonthlyCharges DESC) as Charge_Rank
FROM churn
ORDER BY Contract, Charge_Rank
LIMIT 20
"""
print(run_query(q5))

    customerID        Contract  MonthlyCharges Churn_Label  Charge_Rank
0   2302-ANTDP  Month-to-month          117.45     Churned            1
1   8016-NCFVO  Month-to-month          116.50      Stayed            2
2   9659-QEQSY  Month-to-month          115.65      Stayed            3
3   4361-BKAXE  Month-to-month          114.50     Churned            4
4   6710-HSJRD  Month-to-month          114.10      Stayed            5
5   9158-VCTQB  Month-to-month          113.60     Churned            6
6   7279-BUYWN  Month-to-month          113.20     Churned            7
7   0771-WLCLA  Month-to-month          112.95      Stayed            8
8   1583-IHQZE  Month-to-month          112.95     Churned            8
9   9481-IEBZY  Month-to-month          112.90      Stayed           10
10  2587-EKXTS  Month-to-month          111.50      Stayed           11
11  7130-YXBRO  Month-to-month          111.45      Stayed           12
12  3292-PBZEJ  Month-to-month          111.40      Stayed      

# INSIGHTS


# 1 Month-to-month customers churn 15x more than two year contracts
# 2 Churned customers paid 21% more monthly than loyal customers
# 3 Churned customers left 2x faster — 18 months vs 38 months
# 4 1,379 high value customers already lost — price not the issue
# 5 Contract type and tenure are strongest churn predictors